In [ ]:
import os
import numpy as np
from matplotlib.lines import Line2D
from scipy.stats import pearsonr, gaussian_kde, linregress, ttest_ind, sem, zscore
from sklearn.metrics import r2_score, mean_squared_error
from scipy.stats import norm
from scipy.stats import percentileofscore
from sklearn.utils.validation import check_random_state
from math import factorial
from more_itertools import distinct_permutations
import statsmodels.api as sm

import matplotlib.pyplot as plt

import pandas as pd
from sklearn.model_selection import KFold, train_test_split, StratifiedKFold
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm
from scipy import stats

import numpy as np
from scipy.stats import linregress
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import KFold, ParameterGrid, train_test_split
from tqdm.notebook import tqdm

import warnings
import random
#from torch.utils.data import SubsetRandomSampler
from sklearn.utils import resample

from sklearn.preprocessing import MinMaxScaler
import re

warnings.filterwarnings("ignore")



## Load data

In [ ]:
data = pd.read_csv('DataShare/BBAG-long.csv', index_col=0)

In [ ]:
data.shape

In [ ]:
vars_ = ['Mono', 'One',	'Two',	'Three', 'Total']


# Odds ratios

In [ ]:
bins = [51, 64, 77, 90]
labels = ['51–64', '65–77', '78–90']

data['AgeGroup'] = pd.cut(data['Age'], bins=bins, labels=labels, include_lowest=True)

df_age_1 = data[data['AgeGroup'] == '51–64']
df_age_2 = data[data['AgeGroup'] == '65–77']
df_age_3 = data[data['AgeGroup'] == '78–90']

print(data['AgeGroup'].value_counts())


In [ ]:
39898 + 31957 + 14294

## Without co-vars

In [ ]:
results_merge_df_all = data.copy()
results_merge_df_all = results_merge_df_all.loc[:, ~results_merge_df_all.columns.duplicated()]


def summarize_from_models(rr_values, ci_low_values, ci_high_values, z_values):
    rr_values = np.array(rr_values)
    ci_low_values = np.array(ci_low_values)
    ci_high_values = np.array(ci_high_values)
    z_values = np.array(z_values)

    rr_mean = np.mean(rr_values)
    ci_low_mean = np.mean(ci_low_values)
    ci_high_mean = np.mean(ci_high_values)
    z_mean = np.mean(z_values)

    return ci_low_mean, ci_high_mean, rr_mean, z_mean


def p_value_from_effect_ci(effect_mean, ci_low, ci_high):
    if effect_mean <= 0 or ci_low <= 0 or ci_high <= 0:
        return np.nan, np.nan

    se_log_effect = (np.log(ci_high) - np.log(ci_low)) / (2 * 1.96)

    if se_log_effect == 0:
        return np.nan, np.nan

    z_combined = np.log(effect_mean) / se_log_effect
    p_combined = 2 * (1 - norm.cdf(abs(z_combined)))

    return z_combined, p_combined


n_iterations = 1000

df_directions_odd = pd.DataFrame()

for i in vars_:

    print(i)

    c_results_merge_df_all = results_merge_df_all.copy()

    c_results_merge_df_all.dropna(
        subset=[i, "delta_time", "GAP_bin", "country", "GAP_corrected"],
        inplace=True
    )

    c_results_merge_df_all.reset_index(drop=True, inplace=True)

    X_resid = sm.add_constant(c_results_merge_df_all[["delta_time"]])

    resid_model = sm.OLS(
        c_results_merge_df_all["GAP_corrected"],
        X_resid
    ).fit()

    c_results_merge_df_all["GAP_corrected"] = resid_model.resid

    c_results_merge_df_all["GAP_bin"] = np.where(
        c_results_merge_df_all["GAP_corrected"] > 0,
        1,
        0
    )

    y_ols = c_results_merge_df_all["GAP_bin"]
    X_ols = c_results_merge_df_all[[i]].copy()

    rr_values = []
    ci_low_values = []
    ci_high_values = []
    z_values = []


    for iteration in range(n_iterations):

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X_ols,
                y_ols,
                test_size=0.2,
                random_state=iteration
            )

            scaler = MinMaxScaler((0.05, 0.95))

            X_train_scaled = scaler.fit_transform(X_train)
            X_train_scaled = pd.DataFrame(
                X_train_scaled,
                columns=X_train.columns,
                index=X_train.index
            )

            X_test_scaled = scaler.transform(X_test)
            X_test_scaled = pd.DataFrame(
                X_test_scaled,
                columns=X_test.columns,
                index=X_test.index
            )

            X_train_scaled["intercept"] = 1
            X_test_scaled["intercept"] = 1

            model = sm.GLM(
                y_train,
                X_train_scaled,
                family=sm.families.Binomial(
                    link=sm.families.links.log()
                )
            ).fit(disp=0)

            params = model.params

            conf = np.exp(model.conf_int())
            conf["RR"] = np.exp(params)
            conf["z"] = model.tvalues
            conf["P>|z|"] = model.pvalues
            conf.columns = ["2.5%", "97.5%", "RR", "z", "P>|z|"]

            rr_values.append(conf.loc[i, "RR"])
            ci_low_values.append(conf.loc[i, "2.5%"])
            ci_high_values.append(conf.loc[i, "97.5%"])
            z_values.append(conf.loc[i, "z"])

        except Exception as e:
            print(f"Iteration {iteration} failed for {i}: {e}")


    if len(rr_values) > 1:

        ci_low, ci_high, rr_mean, z_mean_original = summarize_from_models(
            rr_values=rr_values,
            ci_low_values=ci_low_values,
            ci_high_values=ci_high_values,
            z_values=z_values
        )

        z_combined, p_combined = p_value_from_effect_ci(
            effect_mean=rr_mean,
            ci_low=ci_low,
            ci_high=ci_high
        )

    else:

        ci_low = np.nan
        ci_high = np.nan
        rr_mean = np.nan
        z_mean_original = np.nan
        z_combined = np.nan
        p_combined = np.nan

    df_temp = pd.DataFrame({
        "2.5%": [ci_low],
        "97.5%": [ci_high],
        "RR": [rr_mean],
        "z": [z_combined],
        "P>|z|": [f"{p_combined:.2e}" if not np.isnan(p_combined) else np.nan],       
        "z_mean_original": [z_mean_original],
        "Feature": [i],
        "n_iterations_ok": [len(rr_values)]
    })

    df_directions_odd = pd.concat(
        [df_directions_odd, df_temp],
        ignore_index=True
    )


df_directions_odd = df_directions_odd.reset_index(drop=True)
df_directions_odd_age = df_directions_odd.reset_index(drop=True)

In [ ]:
results_merge_df_all = df_age_1.copy()
results_merge_df_all = results_merge_df_all.loc[:, ~results_merge_df_all.columns.duplicated()]


def summarize_from_models(rr_values, ci_low_values, ci_high_values, z_values):
    rr_values = np.array(rr_values)
    ci_low_values = np.array(ci_low_values)
    ci_high_values = np.array(ci_high_values)
    z_values = np.array(z_values)

    rr_mean = np.mean(rr_values)
    ci_low_mean = np.mean(ci_low_values)
    ci_high_mean = np.mean(ci_high_values)
    z_mean = np.mean(z_values)

    return ci_low_mean, ci_high_mean, rr_mean, z_mean


def p_value_from_effect_ci(effect_mean, ci_low, ci_high):
    if effect_mean <= 0 or ci_low <= 0 or ci_high <= 0:
        return np.nan, np.nan

    se_log_effect = (np.log(ci_high) - np.log(ci_low)) / (2 * 1.96)

    if se_log_effect == 0:
        return np.nan, np.nan

    z_combined = np.log(effect_mean) / se_log_effect
    p_combined = 2 * (1 - norm.cdf(abs(z_combined)))

    return z_combined, p_combined


n_iterations = 1000

df_directions_odd = pd.DataFrame()

for i in vars_:

    print(i)

    c_results_merge_df_all = results_merge_df_all.copy()

    c_results_merge_df_all.dropna(
        subset=[i, "delta_time", "GAP_bin", "country", "GAP_corrected"],
        inplace=True
    )

    c_results_merge_df_all.reset_index(drop=True, inplace=True)

    X_resid = sm.add_constant(c_results_merge_df_all[["delta_time"]])

    resid_model = sm.OLS(
        c_results_merge_df_all["GAP_corrected"],
        X_resid
    ).fit()

    c_results_merge_df_all["GAP_corrected"] = resid_model.resid

    c_results_merge_df_all["GAP_bin"] = np.where(
        c_results_merge_df_all["GAP_corrected"] > 0,
        1,
        0
    )

    y_ols = c_results_merge_df_all["GAP_bin"]
    X_ols = c_results_merge_df_all[[i]].copy()

    rr_values = []
    ci_low_values = []
    ci_high_values = []
    z_values = []


    for iteration in range(n_iterations):

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X_ols,
                y_ols,
                test_size=0.2,
                random_state=iteration
            )

            scaler = MinMaxScaler((0.05, 0.95))

            X_train_scaled = scaler.fit_transform(X_train)
            X_train_scaled = pd.DataFrame(
                X_train_scaled,
                columns=X_train.columns,
                index=X_train.index
            )

            X_test_scaled = scaler.transform(X_test)
            X_test_scaled = pd.DataFrame(
                X_test_scaled,
                columns=X_test.columns,
                index=X_test.index
            )

            X_train_scaled["intercept"] = 1
            X_test_scaled["intercept"] = 1

            model = sm.GLM(
                y_train,
                X_train_scaled,
                family=sm.families.Binomial(
                    link=sm.families.links.log()
                )
            ).fit(disp=0)

            params = model.params

            conf = np.exp(model.conf_int())
            conf["RR"] = np.exp(params)
            conf["z"] = model.tvalues
            conf["P>|z|"] = model.pvalues
            conf.columns = ["2.5%", "97.5%", "RR", "z", "P>|z|"]

            rr_values.append(conf.loc[i, "RR"])
            ci_low_values.append(conf.loc[i, "2.5%"])
            ci_high_values.append(conf.loc[i, "97.5%"])
            z_values.append(conf.loc[i, "z"])

        except Exception as e:
            print(f"Iteration {iteration} failed for {i}: {e}")


    if len(rr_values) > 1:

        ci_low, ci_high, rr_mean, z_mean_original = summarize_from_models(
            rr_values=rr_values,
            ci_low_values=ci_low_values,
            ci_high_values=ci_high_values,
            z_values=z_values
        )

        z_combined, p_combined = p_value_from_effect_ci(
            effect_mean=rr_mean,
            ci_low=ci_low,
            ci_high=ci_high
        )

    else:

        ci_low = np.nan
        ci_high = np.nan
        rr_mean = np.nan
        z_mean_original = np.nan
        z_combined = np.nan
        p_combined = np.nan

    df_temp = pd.DataFrame({
        "2.5%": [ci_low],
        "97.5%": [ci_high],
        "RR": [rr_mean],
        "z": [z_combined],
        "P>|z|": [f"{p_combined:.2e}" if not np.isnan(p_combined) else np.nan],       
        "z_mean_original": [z_mean_original],
        "Feature": [i],
        "n_iterations_ok": [len(rr_values)]
    })

    df_directions_odd = pd.concat(
        [df_directions_odd, df_temp],
        ignore_index=True
    )


df_directions_odd = df_directions_odd.reset_index(drop=True)

df_directions_odd_age_group_01 = df_directions_odd.reset_index(drop=True)


In [ ]:
results_merge_df_all = df_age_2.copy()
results_merge_df_all = results_merge_df_all.loc[:, ~results_merge_df_all.columns.duplicated()]


def summarize_from_models(rr_values, ci_low_values, ci_high_values, z_values):
    rr_values = np.array(rr_values)
    ci_low_values = np.array(ci_low_values)
    ci_high_values = np.array(ci_high_values)
    z_values = np.array(z_values)

    rr_mean = np.mean(rr_values)
    ci_low_mean = np.mean(ci_low_values)
    ci_high_mean = np.mean(ci_high_values)
    z_mean = np.mean(z_values)

    return ci_low_mean, ci_high_mean, rr_mean, z_mean


def p_value_from_effect_ci(effect_mean, ci_low, ci_high):
    if effect_mean <= 0 or ci_low <= 0 or ci_high <= 0:
        return np.nan, np.nan

    se_log_effect = (np.log(ci_high) - np.log(ci_low)) / (2 * 1.96)

    if se_log_effect == 0:
        return np.nan, np.nan

    z_combined = np.log(effect_mean) / se_log_effect
    p_combined = 2 * (1 - norm.cdf(abs(z_combined)))

    return z_combined, p_combined


n_iterations = 1000

df_directions_odd = pd.DataFrame()

for i in vars_:

    print(i)

    c_results_merge_df_all = results_merge_df_all.copy()

    c_results_merge_df_all.dropna(
        subset=[i, "delta_time", "GAP_bin", "country", "GAP_corrected"],
        inplace=True
    )

    c_results_merge_df_all.reset_index(drop=True, inplace=True)

    X_resid = sm.add_constant(c_results_merge_df_all[["delta_time"]])

    resid_model = sm.OLS(
        c_results_merge_df_all["GAP_corrected"],
        X_resid
    ).fit()

    c_results_merge_df_all["GAP_corrected"] = resid_model.resid

    c_results_merge_df_all["GAP_bin"] = np.where(
        c_results_merge_df_all["GAP_corrected"] > 0,
        1,
        0
    )

    y_ols = c_results_merge_df_all["GAP_bin"]
    X_ols = c_results_merge_df_all[[i]].copy()

    rr_values = []
    ci_low_values = []
    ci_high_values = []
    z_values = []


    for iteration in range(n_iterations):

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X_ols,
                y_ols,
                test_size=0.2,
                random_state=iteration
            )

            scaler = MinMaxScaler((0.05, 0.95))

            X_train_scaled = scaler.fit_transform(X_train)
            X_train_scaled = pd.DataFrame(
                X_train_scaled,
                columns=X_train.columns,
                index=X_train.index
            )

            X_test_scaled = scaler.transform(X_test)
            X_test_scaled = pd.DataFrame(
                X_test_scaled,
                columns=X_test.columns,
                index=X_test.index
            )

            X_train_scaled["intercept"] = 1
            X_test_scaled["intercept"] = 1

            model = sm.GLM(
                y_train,
                X_train_scaled,
                family=sm.families.Binomial(
                    link=sm.families.links.log()
                )
            ).fit(disp=0)

            params = model.params

            conf = np.exp(model.conf_int())
            conf["RR"] = np.exp(params)
            conf["z"] = model.tvalues
            conf["P>|z|"] = model.pvalues
            conf.columns = ["2.5%", "97.5%", "RR", "z", "P>|z|"]

            rr_values.append(conf.loc[i, "RR"])
            ci_low_values.append(conf.loc[i, "2.5%"])
            ci_high_values.append(conf.loc[i, "97.5%"])
            z_values.append(conf.loc[i, "z"])

        except Exception as e:
            print(f"Iteration {iteration} failed for {i}: {e}")


    if len(rr_values) > 1:

        ci_low, ci_high, rr_mean, z_mean_original = summarize_from_models(
            rr_values=rr_values,
            ci_low_values=ci_low_values,
            ci_high_values=ci_high_values,
            z_values=z_values
        )

        z_combined, p_combined = p_value_from_effect_ci(
            effect_mean=rr_mean,
            ci_low=ci_low,
            ci_high=ci_high
        )

    else:

        ci_low = np.nan
        ci_high = np.nan
        rr_mean = np.nan
        z_mean_original = np.nan
        z_combined = np.nan
        p_combined = np.nan

    df_temp = pd.DataFrame({
        "2.5%": [ci_low],
        "97.5%": [ci_high],
        "RR": [rr_mean],
        "z": [z_combined],
        "P>|z|": [f"{p_combined:.2e}" if not np.isnan(p_combined) else np.nan],       
        "z_mean_original": [z_mean_original],
        "Feature": [i],
        "n_iterations_ok": [len(rr_values)]
    })

    df_directions_odd = pd.concat(
        [df_directions_odd, df_temp],
        ignore_index=True
    )


df_directions_odd = df_directions_odd.reset_index(drop=True)

df_directions_odd_age_group_02 = df_directions_odd.reset_index(drop=True)


In [ ]:
results_merge_df_all = df_age_3.copy()
results_merge_df_all = results_merge_df_all.loc[:, ~results_merge_df_all.columns.duplicated()]


def summarize_from_models(rr_values, ci_low_values, ci_high_values, z_values):
    rr_values = np.array(rr_values)
    ci_low_values = np.array(ci_low_values)
    ci_high_values = np.array(ci_high_values)
    z_values = np.array(z_values)

    rr_mean = np.mean(rr_values)
    ci_low_mean = np.mean(ci_low_values)
    ci_high_mean = np.mean(ci_high_values)
    z_mean = np.mean(z_values)

    return ci_low_mean, ci_high_mean, rr_mean, z_mean


def p_value_from_effect_ci(effect_mean, ci_low, ci_high):
    if effect_mean <= 0 or ci_low <= 0 or ci_high <= 0:
        return np.nan, np.nan

    se_log_effect = (np.log(ci_high) - np.log(ci_low)) / (2 * 1.96)

    if se_log_effect == 0:
        return np.nan, np.nan

    z_combined = np.log(effect_mean) / se_log_effect
    p_combined = 2 * (1 - norm.cdf(abs(z_combined)))

    return z_combined, p_combined


n_iterations = 1000

df_directions_odd = pd.DataFrame()

for i in vars_:

    print(i)

    c_results_merge_df_all = results_merge_df_all.copy()

    c_results_merge_df_all.dropna(
        subset=[i, "delta_time", "GAP_bin", "country", "GAP_corrected"],
        inplace=True
    )

    c_results_merge_df_all.reset_index(drop=True, inplace=True)

    X_resid = sm.add_constant(c_results_merge_df_all[["delta_time"]])

    resid_model = sm.OLS(
        c_results_merge_df_all["GAP_corrected"],
        X_resid
    ).fit()

    c_results_merge_df_all["GAP_corrected"] = resid_model.resid

    c_results_merge_df_all["GAP_bin"] = np.where(
        c_results_merge_df_all["GAP_corrected"] > 0,
        1,
        0
    )

    y_ols = c_results_merge_df_all["GAP_bin"]
    X_ols = c_results_merge_df_all[[i]].copy()

    rr_values = []
    ci_low_values = []
    ci_high_values = []
    z_values = []


    for iteration in range(n_iterations):

        try:
            X_train, X_test, y_train, y_test = train_test_split(
                X_ols,
                y_ols,
                test_size=0.2,
                random_state=iteration
            )

            scaler = MinMaxScaler((0.05, 0.95))

            X_train_scaled = scaler.fit_transform(X_train)
            X_train_scaled = pd.DataFrame(
                X_train_scaled,
                columns=X_train.columns,
                index=X_train.index
            )

            X_test_scaled = scaler.transform(X_test)
            X_test_scaled = pd.DataFrame(
                X_test_scaled,
                columns=X_test.columns,
                index=X_test.index
            )

            X_train_scaled["intercept"] = 1
            X_test_scaled["intercept"] = 1

            model = sm.GLM(
                y_train,
                X_train_scaled,
                family=sm.families.Binomial(
                    link=sm.families.links.log()
                )
            ).fit(disp=0)

            params = model.params

            conf = np.exp(model.conf_int())
            conf["RR"] = np.exp(params)
            conf["z"] = model.tvalues
            conf["P>|z|"] = model.pvalues
            conf.columns = ["2.5%", "97.5%", "RR", "z", "P>|z|"]

            rr_values.append(conf.loc[i, "RR"])
            ci_low_values.append(conf.loc[i, "2.5%"])
            ci_high_values.append(conf.loc[i, "97.5%"])
            z_values.append(conf.loc[i, "z"])

        except Exception as e:
            print(f"Iteration {iteration} failed for {i}: {e}")


    if len(rr_values) > 1:

        ci_low, ci_high, rr_mean, z_mean_original = summarize_from_models(
            rr_values=rr_values,
            ci_low_values=ci_low_values,
            ci_high_values=ci_high_values,
            z_values=z_values
        )

        z_combined, p_combined = p_value_from_effect_ci(
            effect_mean=rr_mean,
            ci_low=ci_low,
            ci_high=ci_high
        )

    else:

        ci_low = np.nan
        ci_high = np.nan
        rr_mean = np.nan
        z_mean_original = np.nan
        z_combined = np.nan
        p_combined = np.nan

    df_temp = pd.DataFrame({
        "2.5%": [ci_low],
        "97.5%": [ci_high],
        "RR": [rr_mean],
        "z": [z_combined],
        "P>|z|": [f"{p_combined:.2e}" if not np.isnan(p_combined) else np.nan],       
        "z_mean_original": [z_mean_original],
        "Feature": [i],
        "n_iterations_ok": [len(rr_values)]
    })

    df_directions_odd = pd.concat(
        [df_directions_odd, df_temp],
        ignore_index=True
    )


df_directions_odd = df_directions_odd.reset_index(drop=True)

df_directions_odd_age_group_03 = df_directions_odd.reset_index(drop=True)


In [ ]:
df_directions_odd_age

In [ ]:
df_directions_odd_age_group_01

In [ ]:
df_directions_odd_age_group_02

In [ ]:
df_directions_odd_age_group_03

In [ ]:
df_mono = pd.DataFrame()

ods_list = [df_directions_odd_age_group_03, df_directions_odd_age_group_02, df_directions_odd_age_group_01]
range_list = ['78–90', '65–77', '51–64']
for i in range(3):
    df_ = ods_list[i].copy()
    fila_mono = df_[df_['Feature'] == 'Mono']
    fila_mono.iloc[0, -2] = 'Mono ' +  range_list[i]
    df_mono = pd.concat([df_mono, fila_mono])


df_one = pd.DataFrame()

ods_list = [df_directions_odd_age_group_03, df_directions_odd_age_group_02, df_directions_odd_age_group_01]
range_list = ['78–90', '65–77', '51–64']
for i in range(3):
    df_ = ods_list[i].copy()
    fila_one = df_[df_['Feature'] == 'One']
    fila_one.iloc[0, -2] = 'One ' +  range_list[i]
    df_one = pd.concat([df_one, fila_one])

df_two = pd.DataFrame()

ods_list = [df_directions_odd_age_group_03, df_directions_odd_age_group_02, df_directions_odd_age_group_01]
range_list = ['78–90', '65–77', '51–64']
for i in range(3):
    df_ = ods_list[i].copy()
    fila_two = df_[df_['Feature'] == 'Two']
    fila_two.iloc[0, -2] = 'Two ' +  range_list[i]
    df_two = pd.concat([df_two, fila_two])


df_three = pd.DataFrame()

ods_list = [df_directions_odd_age_group_03, df_directions_odd_age_group_02, df_directions_odd_age_group_01]
range_list = ['78–90', '65–77', '51–64']
for i in range(3):
    df_ = ods_list[i].copy()
    fila_three = df_[df_['Feature'] == 'Three']
    fila_three.iloc[0, -2] = 'Three ' +  range_list[i]
    df_three = pd.concat([df_three, fila_three])


df_total = pd.DataFrame()

ods_list = [df_directions_odd_age_group_03, df_directions_odd_age_group_02, df_directions_odd_age_group_01]
range_list = ['78–90', '65–77', '51–64']
for i in range(3):
    df_ = ods_list[i].copy()
    fila_total = df_[df_['Feature'] == 'Total']
    fila_total.iloc[0, -2] = 'Total ' +  range_list[i]
    df_total = pd.concat([df_total, fila_total])

In [ ]:
df_list = [df_mono, df_one, df_two, df_three, df_total] 

In [ ]:
df_mono

In [ ]:
plt.figure(figsize=(7.5, 3))

count = 2
for i in df_list:
    df_directions_odd = i.copy()
    plt.subplot(2,3,count)
    plt.errorbar(df_directions_odd['RR'], df_directions_odd['Feature'], 
                 xerr=[df_directions_odd['RR'] - df_directions_odd['2.5%'], df_directions_odd['97.5%'] - df_directions_odd['RR']], 
                 fmt='none', c='k', capsize=5)
    
    # Añadir los puntos de los coeficientes
    plt.scatter(df_directions_odd['RR'], df_directions_odd['Feature'], s=10, color='red', zorder=10)
    plt.axvline(x=1, color='red', linestyle='--', linewidth=2, label='x = 1')
    
    
    plt.xlim([0,2.0])
    plt.ylim([-0.5,2.5])
    plt.title('RR [95% CI]', fontsize = 9)
    plt.xlabel('Relative Risk')
    plt.ylabel('Feature')

    text = df_directions_odd.reset_index().loc[0, 'Feature']
    match = re.match(r'^\S+', text)
    name = match.group()

    df_directions_odd['RR'] = df_directions_odd['RR'].round(2)
    df_directions_odd['z'] = df_directions_odd['z'].round(2)
    
    df_directions_odd['CI'] = df_directions_odd['2.5%'].round(2).astype(str) + " – " + df_directions_odd['97.5%'].round(2).astype(str)
    
    df_directions_odd['p_formatted'] = df_directions_odd['P>|z|']
    
    df_formatted = df_directions_odd[['Feature', 'RR', 'CI', 'z', 'p_formatted']].copy()
    df_formatted['age_start'] = df_formatted['Feature'].str.extract(r'(\d{2})')[0].astype(int)

    
    df_formatted.columns = ['Feature','RR', '95% CI', 'z', 'p-value', 'age_start']
    df_formatted = df_formatted.sort_values(by='age_start').drop(columns='age_start').reset_index(drop=True)
    display(df_formatted)
    df_formatted.to_excel('Results/RR_long_ranged_' + name + '.xlsx')
    count+=1

plt.tight_layout()

plt.savefig('Figures/long_RR_age-ranged.pdf', format='pdf', bbox_inches='tight', dpi=300)

In [ ]:
df_directions_odd_age_group_01

